<a href="https://colab.research.google.com/github/EmanuelePinci/AML-project-IRIDE/blob/Emanuele/Medica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Setup software
import torch
print(torch.__version__)


#install pytorch geometric libraries in google colab
#!pip install torch-geometric
!pip uninstall torch-scatter torch-sparse torch-geometric torch-cluster torch-spline-conv --y
!pip install torch-scatter -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-spline-conv -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-sparse -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install torch-cluster -f https://data.pyg.org/whl/torch-{torch.__version__}.html
!pip install git+https://github.com/pyg-team/pytorch_geometric.git


2.10.0+cpu
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 10.4 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.9/306.9 kB 6.0 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.2 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cpu.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 828.2/828.2 kB 11.2 MB/s eta 0:00:00
  Cloning https://github.com/pyg-team/pytorch_geometric.git to /tmp/pip-req-build-gz51kazn
  Running command git clone --filter=blob:none --quiet https://github.com/pyg-team/pytorch_geometric.git /tmp/pip-req-build-gz51kazn
  Resolved https://github.com/pyg-team/pytorch_geometric.git to commit a5b69c37a05561ebb92931b3d586d664a7269585
  Installing build dependencies ... done
  Getting requirements to buil

In [ ]:
#@title 2. Import libraries
import math, torch, numpy as np, matplotlib.pyplot as plt, os, pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split # Usiamo solo random_split da qui
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder #per gestire le parti dell'occhio
from torch.optim.lr_scheduler import CosineAnnealingLR

# --- IMPORT SPECIFICI PER PYTORCH GEOMETRIC ---
import torch_geometric
from torch_geometric.data import Data, Dataset # Strutture dati per i grafi
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, SAGEConv # Esempi di layer convoluzionali per grafi

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch', torch.__version__, '| PyG', torch_geometric.__version__, '| device →', device)

Torch 2.10.0+cpu | PyG 2.8.0 | device → cpu


In [ ]:
#@title Codice per convertire i file delle simulazioni in npz

# Immaginiamo di avere i dati pronti da 10.000 simulazioni
# Ogni simulazione ha 216.000 voxel
num_sim = 10000
num_voxel = 216000

# Creiamo matrici vuote (o riempite leggendo i tuoi file reali)
# Features: [Simulazioni, Voxel, 4 colonne (Anat, X, Y, Z)]
features_matrix = np.zeros((num_sim, num_voxel, 4), dtype=object)

# Doses: [Simulazioni, Voxel]
doses_matrix = np.zeros((num_sim, num_voxel), dtype=np.float32)

# --- QUI ANDREBBE IL CICLO PER RIEMPIRLE DAI .TXT ---
# for i in range(num_sim):
#     dati_txt = caricamento_da_file(f'sim_{i}.txt')
#     features_matrix[i] = dati_txt[:, 1:5]
#     doses_matrix[i] = dati_txt[:, 5]

# SALVATAGGIO FINALE
np.savez_compressed('Datisimulazione.npz',
                    features=features_matrix,
                    doses=doses_matrix)
#Codice conversione da CSV a npz
all_features = []
all_doses = []

# Definiamo il percorso base
base_path = 'percorso'

print("Inizio elaborazione...")

for i in range(10000):
    file_path = f'{base_path}/sim_{i}.csv'

    # Controllo esistenza file (evita crash se manca una simulazione)
    if not os.path.exists(file_path):
        continue

    # Caricamento ottimizzato: leggiamo solo le colonne che servono
    # Supponendo: col 1-4 (Features), col 5 (Dose), col specifica per 'anatomia'
    # Se 'anatomia' è una delle prime 4, adattiamo usecols
    try:
        df = pd.read_csv(file_path)

        # Filtro aria
        df = df[df['anatomia'] != 'aria']

        # Estrazione (Assicurati che gli indici siano corretti dopo il filtro)
        # .astype('float32') riduce drasticamente il peso dei dati senza perdere precisione utile
        features = df.iloc[:, 1:5].values.astype('float32')
        dose = (df.iloc[:, 5].values * 1e7).astype('float32')

        all_features.append(features)
        all_doses.append(dose)

    except Exception as e:
        print(f"Errore nel file {i}: {e}")

    # Feedback ogni 500 file
    if i % 500 == 0:
        print(f"Processati {i}/10000 file...")

print("Salvataggio in corso (operazione lenta per 10k file)...")

# Salvataggio con allow_pickle=True perché le simulazioni potrebbero
# avere lunghezze diverse (numero di voxel variabile dopo il filtro)
np.savez_compressed('Dati_Pronti_GNN.npz',
                    features=np.array(all_features, dtype=object),
                    doses=np.array(all_doses, dtype=object))

print("Completato!")

In [ ]:
#@title 3. Load Dataset, Preprocessing & Splits

# 1. CARICAMENTO DATI
# allow_pickle=True è fondamentale perché la colonna anatomia contiene parole (stringhe)
!wget -N Datisimulazione.npz #non ho ancora i dati della simulazione quindi ho chiamto un file immaginario
npz = np.load('Datisimulazione.npz', allow_pickle=True)

# Assumiamo che npz['features'] abbia 5 colonne: [id_voxel, anatomia, x, y, z]
# Scartiamo SUBITO la colonna 0 (id_voxel) perché non ha senso fisico
X_raw = npz['features'][:, :, 1:] # Ora X_raw ha 4 colonne: [anatomia, x, y, z]

# La dose è il nostro TARGET (y), la teniamo rigorosamente separata!
# Moltiplichiamo per 10^7 (numero eventi) per evitare numeri troppo piccoli (Vanishing Gradient)
y_all = npz['doses'].astype('float32') * 1e7 #chiedere 10 cifre significative

# 2. SPLIT SULLE SIMULAZIONI (Evitiamo il Data Leakage spaziale)
n_simulazioni = X_raw.shape[0]
idx = np.random.permutation(n_simulazioni)
X_all = X_raw[idx]
y_all = y_all[idx]

n_train = int(0.8 * n_simulazioni)
n_val   = int(0.1 * n_simulazioni)
n_test  = n_simulazioni - n_train - n_val

# Creiamo i set temporanei
X_train_tmp = X_all[:n_train]
y_train =  y_all[:n_train]
X_val_tmp = X_all[n_train:n_train+n_val]
y_val     = y_all[n_train:n_train+n_val]
X_test_tmp = X_all[n_train+n_val:]
y_test   = y_all[n_train+n_val:]

print(f'Split sizes (Simulazioni): Train={n_train}, Val={n_val}, Test={n_test}')

#arrivato qui, per il momento so cosa ho fatto e perché è stato fatto

# 3. ONE-HOT ENCODING (Anatomia) E SCALING (Coordinate)
# Colonna 0 = anatomia, Colonne 1,2,3 = x,y,z
anatomia_train = X_train_tmp[0, :, 0].reshape(-1, 1) # L'anatomia è fissa, usiamo il primo grafo

# Fit dell'Encoder sulle parole
encoder = OneHotEncoder(sparse_output=False).fit(anatomia_train)
categorie = encoder.categories_[0]
print(f"Categorie anatomiche trovate: {categorie}")

# Trasformiamo le parole in 0 e 1 (Shape: N_voxel, N_categorie)
anatomia_encoded = encoder.transform(anatomia_train).astype('float32')

# Fit dello Scaler SOLO sulle coordinate del Training Set
X_train_coords = X_train_tmp[:, :, 1:4].reshape(-1, 3).astype('float32')
scaler = StandardScaler().fit(X_train_coords)

def processa_features(X_set, scaler, anatomia_encoded):
    # Scala le coordinate
    shape_coords = X_set[:, :, 1:4].shape
    coords_scaled = scaler.transform(X_set[:, :, 1:4].reshape(-1, 3)).reshape(shape_coords)

    # Replica l'anatomia codificata per ogni simulazione nel batch
    anatomia_espansa = np.tile(anatomia_encoded, (X_set.shape[0], 1, 1))

    # Unisce [Anatomia_OneHot, X_scaled, Y_scaled, Z_scaled]
    return np.concatenate([anatomia_espansa, coords_scaled], axis=-1).astype('float32')

X_train_proc = processa_features(X_train_tmp, scaler, anatomia_encoded)
X_val_proc   = processa_features(X_val_tmp, scaler, anatomia_encoded)
X_test_proc  = processa_features(X_test_tmp, scaler, anatomia_encoded)

# 4. RIMOZIONE DELL'ARIA (Ottimizzazione computazionale)
# Troviamo l'indice della colonna "Aria" (o "Vuoto", "Air") generata dall'encoder
# Sostituisci 'Aria' con il nome esatto che appare nel print delle categorie
#indice_aria = np.where(categorie == 'Aria')[0][0]
#maschera_occhio = X_train_proc[0, :, indice_aria] == 0

#X_train = X_train_proc[:, maschera_occhio, :]
#y_train = y_train[:, maschera_occhio]
#X_val   = X_val_proc[:, maschera_occhio, :]
#y_val   = y_val[:, maschera_occhio]
#X_test  = X_test_proc[:, maschera_occhio, :]
#y_test  = y_test[:, maschera_occhio]

print(f"\nNormalizzazione completata! Voxel ridotti da {len(maschera_occhio)} a {maschera_occhio.sum()} per grafo.")
print(f"Shape finale X_train: {X_train.shape} (Simulazioni, Voxel validi, Features totali)")

--2026-05-05 16:01:16--  http://collagamentoalfile.npz/
Resolving collagamentoalfile.npz (collagamentoalfile.npz)... failed: Name or service not known.
wget: unable to resolve host address ‘collagamentoalfile.npz’


FileNotFoundError: [Errno 2] No such file or directory: 'file.npz'

In [ ]:
#CIAO SONO EMA